In [ ]:
A = [0.5 0.5 0.5 ; 0.25 0.0 0.25 ; 0.25 0.5 0.5]
xtoday = [1; 0; 0]

for k in 1:50
    println(k)
    xtomorrow = A * xtoday
    xtoday = xtomorrow
    println(xtomorrow, digits = 3)
end



In [ ]:
using QuantumToolbox
using GLMakie

H = 0.5 * sigmay()
state = basis(2,0)
e_ops = [
    proj(basis(2,0)),
    proj(basis(2,1)),
    basis(2,0) * basis(2,1)'
]
time_pnts = LinRange(0,10,100)
sol = sesolve(H, state, time_pnts, e_ops = e_ops, progress_bar = Val(false))
times = sol.times
print(sol)
print(times)




fig

In [ ]:
using QuantumToolbox
using GLMakie

H = 0.5 * sigmaz()
state = basis(2, 0)
e_ops = [
    proj(basis(2, 0)),
    proj(basis(2, 1)),
    basis(2, 0) * basis(2, 1)'
]
time_pnts = LinRange(0, 5, 100)
sol = sesolve(H, state, time_pnts; e_ops = e_ops, progress_bar = Val(false))

times = sol.times
expt1 = real(sol.expect[1, :])
expt2 = real(sol.expect[2, :])
expt3 = real(sol.expect[3, :])

fig = Figure(size = (500, 350))
ax = Axis(fig[1, 1], xlabel = L"t", ylabel = L"\langle \rho(t) \rangle")
lines!(ax, times, expt1, label = L"\langle 0|\rho(t)|0\rangle")
lines!(ax, times, expt2, label = L"\langle 1|\rho(t)|1\rangle")
lines!(ax, times, expt3, label = L"\langle 0|\rho(t)|1\rangle")
Makie.ylims!(ax, (-0.5, 1.0))
axislegend(ax, position = :lb)
title!("Quantum Evolution")

fig

In [ ]:
using QuantumToolbox
using GLMakie

H = 0.5 * sigmax()
state = basis(2, 0)
time_pnts = LinRange(0, 5, 100)
sol = mesolve(H, state, time_pnts; e_ops = [sigmax()])

times = sol.state
expt1 = real(sol.expect[1, :])
expt2 = real(sol.expect[2, :])
expt3 = real(sol.expect[3, :])

fig = Figure(size = (500, 350))
ax = Axis(fig[1, 1], xlabel = L"t", ylabel = L"\langle \rho(t) \rangle")
lines!(ax, times, expt1, label = L"\langle 0|\rho(t)|0\rangle")
lines!(ax, times, expt2, label = L"\langle 1|\rho(t)|1\rangle")
lines!(ax, times, expt3, label = L"\langle 0|\rho(t)|1\rangle")
Makie.ylims!(ax, (-0.5, 1.0))
axislegend(ax, position = :lb)
title!("Quantum Evolution")

fig

In [ ]:
using QuantumToolbox
using DifferentialEquations
using GLMakie

const γ = 0.1
 


function bloch_Y!(du, u, p, t)
    g = p
    du[1] = -0.2g * u[1]
    du[2] = 0.0
    du[3] = -0.2g * u[3]
end

function bloch_Z!(du, u, p, t)
    g = p
    du[1] = -2g * u[1]
    du[2] = -2g * u[2]
    du[3] = 0.0
end

function bloch_X!(du, u, p, t)
    g = p
    du[1] = 0.0
    du[2] = -2g * u[2]
    du[3] = -2g * u[3]
end

u0 = [1.0, 0.0, 0.0]
tspan = (0.0, 20.0)
tsave = range(tspan[1], tspan[2], length = 150)


prob_X = ODEProblem(bloch_X!, u0, tspan, γ)
prob_Y = ODEProblem(bloch_Y!, u0, tspan, γ)
prob_Z = ODEProblem(bloch_Z!, u0, tspan, γ)

sol_X = solve(prob_X, Tsit5(), saveat = tsave)
sol_Y = solve(prob_Y, Tsit5(), saveat = tsave)
sol_Z = solve(prob_Z, Tsit5(), saveat = tsave)

function animate(sol, filename::String)
    xs = sol[1, :]
    ys = sol[2, :]
    zs = sol[3, :]

    b = Bloch()
    b.view = [50, 30]
    fig, lscene = render(b)
    #println(length(sol.t))

    record(fig, filename, eachindex(sol.t); framerate = 30) do idx
        empty!(fig)
        add_vectors!(b, [xs[idx], ys[idx], zs[idx]])
        add_points!(b, [xs[1:idx], ys[1:idx], zs[1:idx]])
        render(b, location = lscene)
    end

    println("Done!")
end

animate(sol_X, "X.mp4")
#animate(sol_Y, "Y.mp4")
#animate(sol_Z, "Z.mp4")





In [ ]:
using QuantumToolbox
using DifferentialEquations
using GLMakie

const γ = 0.1
 


function bloch_Y!(du, u, p, t)
    g = p
    du[1] = 2*u[3] - 2g*u[1]
    du[2] = 0.0
    du[3] = -2*u[1] - 2g*u[3]
end

function bloch_Z!(du, u, p, t)
    g = p
    du[1] = -2*u[2] - 2g*u[1]
    du[2] = 2*u[1] - 2g*u[2]
    du[3] = 0.0
end

function bloch_X!(du, u, p, t)
    g = p
    du[1] = 0.0
    du[2] = (-2.0)*u[3] - 2.0g*u[2]
    du[3] = 2.0*u[2] - 2.0g*u[3]
end

u0 = [1/sqrt(3), 1/sqrt(3), 1/sqrt(3)]
tspan = (0.0, 20.0)
tsave = range(tspan[1], tspan[2], length = 150)


prob_X = ODEProblem(bloch_X!, u0, tspan, γ)
prob_Y = ODEProblem(bloch_Y!, u0, tspan, γ)
prob_Z = ODEProblem(bloch_Z!, u0, tspan, γ)

sol_X = solve(prob_X, Tsit5(), saveat = tsave)
sol_Y = solve(prob_Y, Tsit5(), saveat = tsave)
sol_Z = solve(prob_Z, Tsit5(), saveat = tsave)

function animate(sol, filename::String)
    xs = sol[1, :]
    ys = sol[2, :]
    zs = sol[3, :]

    b = Bloch()
    b.view = [50, 30]
    fig, lscene = render(b)
    #println(length(sol.t))

    vector_points = Observable([Point3f(0,0,0), Point3f(xs[1], ys[1], zs[1])])
    lines!(lscene, vector_points, color = :pink, linewidth = 5)
    trail_points = Observable([Point3f(0,0,0), Point3f(xs[1], ys[1], zs[1])])
    scatter!(lscene, trail_points, color = :blue, markersize = 6)
    record(fig, filename, eachindex(sol.t); framerate = 30) do idx
        vector_points[] = [Point3f(0,0,0), Point3f(xs[idx], ys[idx], zs[idx])]
        trail_points[] = push!(trail_points[], Point3f(xs[idx], ys[idx], zs[idx]))
    end

    println("Done!")
end

animate(sol_X, "X.mp4")
animate(sol_Y, "Y.mp4")
animate(sol_Z, "Z.mp4")





In [ ]:
using QuantumToolbox
using DifferentialEquations
using GLMakie

const γ = 0.1
 


function bloch_Y!(du, u, p, t)
    g = p
    du[1] = 2*u[3] - 2g*u[1]
    du[2] = 0.0
    du[3] = -2*u[1] - 2g*u[3]
end

function bloch_Z!(du, u, p, t)
    g = p
    du[1] = -2*u[2] - 2g*u[1]
    du[2] = 2*u[1] - 2g*u[2]
    du[3] = 0.0
end

function bloch_X!(du, u, p, t)
    g = p
    du[1] = 0.0
    du[2] = (-2.0)*u[3] - 2.0g*u[2]
    du[3] = 2.0*u[2] - 2.0g*u[3]
end

u0 = [1/sqrt(3), 1/sqrt(3), 1/sqrt(3)]
tspan = (0.0, 20.0)
tsave = range(tspan[1], tspan[2], length = 150)


prob_X = ODEProblem(bloch_X!, u0, tspan, γ)
prob_Y = ODEProblem(bloch_Y!, u0, tspan, γ)
prob_Z = ODEProblem(bloch_Z!, u0, tspan, γ)

sol_X = solve(prob_X, Tsit5(), saveat = tsave)
sol_Y = solve(prob_Y, Tsit5(), saveat = tsave)
sol_Z = solve(prob_Z, Tsit5(), saveat = tsave)

function animate(sol, filename::String)
    xs = sol[1, :]
    ys = sol[2, :]
    zs = sol[3, :]

    b = Bloch()
    b.view = [50, 30]
    fig, lscene = render(b)
    #println(length(sol.t))

    origin = Observable([Point3f(0,0,0)])
    direction = Observable([Point3f(xs[1], ys[1], zs[1])])
    arrows3d!(lscene, origin, direction, color = :pink, shaftradius = 0.02, tipradius = 0.03, tiplength = 0.15)
    trail_points = Observable([Point3f(0,0,0), Point3f(xs[1], ys[1], zs[1])])
    scatter!(lscene, trail_points, color = :blue, markersize = 6)
    record(fig, filename, eachindex(sol.t); framerate = 30) do idx
        direction[] = [Point3f(xs[idx], ys[idx], zs[idx])]
        trail_points[] = push!(trail_points[], Point3f(xs[idx], ys[idx], zs[idx]))
    end

    println("Done!")
end

animate(sol_X, "X.mp4")
animate(sol_Y, "Y.mp4")
animate(sol_Z, "Z.mp4")





Done!
Done!
Done!
